# KALIA v0.2.0 — the registered final evaluation

Session 3 completed the full 4,770-step schedule. This kernel produces the two
numbers the pre-registration actually asks for, on the final checkpoint:

| ID | Question | Threshold |
|---|---|---|
| P-A | matches v0.1.2 on the same yardstick | val loss ≤ **3.0533** |
| P-B | a real gain, above eval noise | val loss ≤ **3.0033** |
| P-C | a real loss | val loss > **3.1033** |
| S-A | no task regresses more than 1.0 pp | benchmark vs v0.1.2 |

Protocol is fixed: `eval_val.py --batches 100 --batch-size 8 --seed 1234`,
819,200 tokens, the same call that produced v0.1.2's 3.0533. P-C is
registered symmetrically with P-B so a regression is reported as one, with no
re-runs, seed-shopping, or protocol changes used to escape it.

CPU kernel: no GPU quota consumed.

In [ ]:
# lm-eval 0.4.9 imports AutoModelForVision2Seq, removed in transformers v5.
!pip install -q "lm-eval==0.4.9" "transformers<5" tiktoken pyyaml datasets
import glob, importlib.metadata as md, os, shutil, sys
for pkg in ('lm_eval', 'transformers', 'torch', 'datasets'):
    try:
        print(f"{pkg:14s} {md.version(pkg)}")
    except Exception as exc:
        print(f"{pkg:14s} MISSING ({exc})")

In [ ]:
root = sorted(glob.glob("/kaggle/input/datasets/subhajitlucky/kalia-code-dev"))[0]
import os
for f in ["eval_val.py", "eval_bench.py", "kalia_lm.py", "model.py",
          "data.py", "eval_reversibility.py"]:
    shutil.copy(f"{root}/{f}", "/kaggle/working/")
    print("copied", f)
os.makedirs("/kaggle/working/eval", exist_ok=True)
for f in glob.glob(f"{root}/eval/*.json"):
    shutil.copy(f, "/kaggle/working/eval/")
os.chdir("/kaggle/working")
ckpt = sorted(glob.glob(
    "/kaggle/input/notebooks/subhajitlucky/kalia-train-v020/**/ckpt.pt",
    recursive=True))
print("checkpoint:", ckpt)
assert ckpt, "attach the kalia-train-v020 kernel output"
CKPT = ckpt[0]
val_bin = sorted(glob.glob(
    "/kaggle/input/notebooks/subhajitlucky/kalia-prep-v2b/**/val.bin",
    recursive=True))
print("val.bin:", val_bin)
assert val_bin, "attach the kalia-prep-v2b kernel output"

## P-A / P-B / P-C — the registered primary protocol

100 batches of 8, seed 1234. The yardstick is the canonical v2b `val.bin`
that v0.1.2 was re-baselined onto at 3.0533, because the corpus rebuild
redefined the held-out set (D43).

In [ ]:
import subprocess
rc = subprocess.run(
    [sys.executable, 'eval_val.py',
     '--ckpt', CKPT,
     '--val-bin', val_bin[0],
     '--batches', '100',
     '--batch-size', '8',
     '--seed', '1234',
     '--out', '/kaggle/working/eval_final.md'],
    capture_output=True, text=True,
)
print(rc.stdout[-2000:])
if rc.returncode != 0:
    print('STDERR:', rc.stderr[-3000:])
assert rc.returncode == 0, 'deterministic eval failed'
print(open('/kaggle/working/eval_final.md').read())

In [ ]:
# eval_val.py writes MARKDOWN, not JSON. Version 1 of this kernel assumed
# JSON and died in json.load with a bare JSONDecodeError, taking the
# benchmark half down with it -- the same 'one step fails and the kernel
# stops' shape as the x18-close failures. So: parse explicitly, and assert
# the parse produced a number rather than trusting a regex to have matched.
import json, re
md = open('/kaggle/working/eval_final.md').read()
m_loss = re.search(r'Val loss:\s*([0-9.]+)', md)
m_bpb = re.search(r'bpB:\s*([0-9.]+)', md)
m_bpt = re.search(r'bytes/token:\s*([0-9.]+)', md)
m_tok = re.search(r'=\s*([0-9,]+)\s*tokens', md)
assert m_loss, f'could not parse val loss from eval_final.md:\n{md}'
assert m_bpb, f'could not parse bpB from eval_final.md:\n{md}'
loss = float(m_loss.group(1))
bpb = float(m_bpb.group(1))
print('bytes/token', m_bpt.group(1) if m_bpt else '?',
      '| tokens', m_tok.group(1) if m_tok else '?')

REF, GOOD, BAD = 3.0533, 3.0033, 3.1033
print()
print('=== registered thresholds (v0.1.2 = 3.0533 on this exact protocol) ===')
print(f'  v0.2.0 val loss {loss:.4f} | bpB {bpb:.4f}')
if loss <= GOOD:
    print(f'  P-B PASS  improvement of {REF-loss:+.4f} nats vs v0.1.2 (needs <= {GOOD})')
    print('  P-A PASS  (P-B implies P-A)')
    print('  P-C not triggered')
elif loss <= REF:
    print(f'  P-A PASS  matches v0.1.2 ({REF-loss:+.4f} nats), inside the noise band')
    print('  P-B FAIL  not >= 0.05 better')
    print('  P-C not triggered')
elif loss <= BAD:
    print(f'  P-A FAIL  {loss-REF:+.4f} nats worse than v0.1.2')
    print('  matched within noise: between 3.0033 and 3.1033')
else:
    print(f'  P-C FAIL  regression of {loss-REF:+.4f} nats, reported as one')
json.dump({'val_loss': loss, 'bpb': bpb, 'batches': 100,
           'batch_size': 8, 'seed': 1234},
          open('/kaggle/working/eval_final.json', 'w'), indent=2)

## S-A — the five zero-shot tasks

This is the half that the interim step-3,470 run left in tension: deterministic
loss improved by 0.169 nats while four of five tasks fell, two of them by more
than the 1.0 pp allowance. The corpus changed underneath that run, so the final
checkpoint is the first one whose benchmark and loss can both be taken at face
value against v0.1.2.

In [ ]:
subprocess.run(
    [sys.executable, 'eval_bench.py',
     '--ckpt', CKPT,
     '--out', '/kaggle/working/bench_final.json',
     '--limit', '500'],
    check=True,
)

In [ ]:
bench = json.load(open('/kaggle/working/bench_final.json'))

# v0.1.2, published, same five tasks, same 0-shot/500-sample protocol.
V012 = {'piqa': 61.4, 'arc_easy': 45.8, 'hellaswag': 36.8,
        'lambada_openai': 23.0, 'winogrande': 50.2}

def acc(res, task):
    m = res[task]
    for key in ('acc_norm,none', 'acc,none'):
        if key in m:
            return m[key] * 100
    raise KeyError(f'{task}: no acc metric, keys={sorted(m)}')

print(f"{'task':18s} {'v0.1.2':>7s} {'v0.2.0':>7s} {'delta':>7s}  S-A")
fails = []
for task, ref in V012.items():
    got = acc(bench, task)
    d = got - ref
    bad = d < -1.0
    if bad:
        fails.append(task)
    print(f"{task:18s} {ref:7.1f} {got:7.1f} {d:+7.2f}  {'FAIL' if bad else 'ok'}")

print()
if fails:
    verdict = 'S-A: FAIL - ' + ', '.join(fails)
else:
    verdict = 'S-A: PASS - no task regressed more than 1.0 pp'
print(verdict)

Read the two verdicts together.

- P-B **and** S-A pass → the compliance rebuild is a genuine, publishable
  improvement and v0.2.0 ships.
- P-B passes, S-A fails → a real loss improvement with a real accuracy cost.
  That is a finding about the corpus, not a model defect, and it says the
  5%→permissive-Python swap is not cost-free — the honest response is to report
  it and stop treating the mixture as inert.
- P-C → a regression. Registered symmetrically, so it is reported as one, with
  no re-runs or protocol changes used to escape it.

The val curve flattened after step 3,250 (2.874–2.940 across the last 1,500
steps), the same J2 plateau that stopped v0.1.2 at 73% of its schedule. Worth
recording either way: this run spent its last 31% of steps inside noise.